# 04 — Layer 3a: Code Mapping (UMLS-CUI Bridge)

🔧 Script · 📚 Reference

Layer 3 starts with code-system unification. Two conditions coded in SNOMED (Synthea) and ICD-10 (Epic) are the same condition — they share a UMLS CUI in our hand-curated crosswalk. This notebook shows the lookup and annotation machinery.


## 1. The hand-curated crosswalk

In [ ]:
from pathlib import Path
import json

ATLAS_ROOT = Path("..").resolve()
CORPUS     = ATLAS_ROOT / "corpus"

xwalk_path = CORPUS / "reference" / "handcrafted-crosswalk" / "showcase.json"
xwalk = json.loads(xwalk_path.read_text())

print(f"Crosswalk version: {xwalk['version']}")
print(f"Entries: {len(xwalk['codes'])}")
print()
for entry in xwalk["codes"][:4]:
    print(f"  {entry['concept_label']}  (CUI {entry['umls_cui']})")
    if entry.get("snomed_ct"):
        print(f"    SNOMED {entry['snomed_ct']['code']} — {entry['snomed_ct']['display']}")
    if entry.get("icd_10_cm"):
        print(f"    ICD-10 {entry['icd_10_cm']['code']} — {entry['icd_10_cm']['display']}")
    if entry.get("rxnorm"):
        print(f"    RxNorm {entry['rxnorm']['rxcui']} — {entry['rxnorm']['display']}")


## 2. lookup_cross — Artifact 1 anchor: Hyperlipidemia

In [ ]:
# 📚 Reference — both codes resolve to CUI C0020473 (Hyperlipidemia)
from ehi_atlas.terminology import lookup_cross
from ehi_atlas.harmonize.code_map import SYS_SNOMED, SYS_ICD10_CM

# Synthea uses SNOMED
snomed_entry = lookup_cross(SYS_SNOMED, "55822004")
print("SNOMED 55822004 →", snomed_entry["concept_label"] if snomed_entry else "not found")
print("  CUI:", snomed_entry["umls_cui"] if snomed_entry else "—")

print()
# Epic projection uses ICD-10
icd10_entry = lookup_cross(SYS_ICD10_CM, "E78.5")
print("ICD-10 E78.5 →", icd10_entry["concept_label"] if icd10_entry else "not found")
print("  CUI:", icd10_entry["umls_cui"] if icd10_entry else "—")


## 3. codings_equivalent

In [ ]:
# 🔧 Script
from ehi_atlas.harmonize.code_map import codings_equivalent

snomed_coding = {"system": SYS_SNOMED,   "code": "55822004", "display": "Hyperlipidemia"}
icd10_coding  = {"system": SYS_ICD10_CM, "code": "E78.5",    "display": "Hyperlipidemia, unspecified"}

result = codings_equivalent(snomed_coding, icd10_coding)
print("codings_equivalent(SNOMED 55822004, ICD-10 E78.5):", result)
print("→ Both map to UMLS CUI C0020473 via the crosswalk")


## 4. annotate_codeable_concept_with_cui

In [ ]:
# 🔧 Script — attaches EXT_UMLS_CUI to each coding in place
import copy
from ehi_atlas.harmonize.code_map import annotate_codeable_concept_with_cui
from ehi_atlas.harmonize.provenance import EXT_UMLS_CUI

sample_cc = {
    "coding": [
        {"system": SYS_SNOMED,   "code": "55822004", "display": "Hyperlipidemia"},
        {"system": SYS_ICD10_CM, "code": "E78.5",    "display": "Hyperlipidemia, unspecified"},
    ],
    "text": "Hyperlipidemia",
}

before_ext = [c.get("extension", []) for c in sample_cc["coding"]]
annotate_codeable_concept_with_cui(sample_cc)
after_ext  = [c.get("extension", []) for c in sample_cc["coding"]]

print("Before: no extensions on codings")
print("After:")
for i, coding in enumerate(sample_cc["coding"]):
    for ext in coding.get("extension", []):
        if ext["url"] == EXT_UMLS_CUI:
            print(f"  coding[{i}] ({coding['system'].split('/')[-1]}) → CUI {ext['valueString']}")


## 5. Note: This is the Artifact 1 anchor mechanism

`annotate_codeable_concept_with_cui` is called on every Condition in every silver bundle during the orchestrator's annotation pass. Once both the Synthea SNOMED and Epic ICD-10 codings carry `EXT_UMLS_CUI = C0020473`, the condition merger clusters them into one gold-tier Condition (notebook 06).


**Next:** [05_layer3_temporal_and_identity.ipynb](./05_layer3_temporal_and_identity.ipynb)